In [ ]:
# want to know if in any podcast or yt video something about AI related topic?
# this RAG based system can answer that

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = ""

In [ ]:
# install libraries
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

### Step 1a - Indexing (Document Ingestion)

For loadiing transcripts in certain language, transcript should be present in that language, otherwise code will break.

In [ ]:
video_id = "" # paste only the ID from video url, not full URL
try:
    # if you don't care which language, this returns the "best" one
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id, langauge= ['en']) # language=['hi'] for hindi transcript,  language ['en'] will through error if video does not contain english transcript

    # Flatten it to plain text
    transcript = " ".join(chunk['text'] for chunk in transcript_list)
    print(transcript)
    
except transcriptsDisabled:
    print("No captions available for this video.")

In [ ]:
transcript_list # transcript based on timestamp

### Step 1b - Indexing (Text Splitting)

In [ ]:
# as video is very long, so we have to split it
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:
chunks[0]

In [ ]:
chunks[100]

### Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Space)

In [ ]:
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
vector_space = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id # ids against each chunk

In [ ]:
vector_store.get_by_ids(['']) # put one id here to see a particular chunk

### Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs = {"k": 4}) # return 4 most similar chunks

In [ ]:
retriever

In [ ]:
retriever.invoke('What is deepmind') # ask qsn related to uploaded video # in retriever input is qsn, op is list of queries

### Step 3 - Augmentation

In [ ]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature = 0.2)

In [ ]:
prompt = PromptTemplate(
    template = """
        You are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.
        
        {context}
        Question:  {question}
        """,
    input_variables = ['context', 'question']
)

In [ ]:
question = "is the topic of aliens discussed in this video? if yes, then what was discussed?"
retrieved_docs = retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

### Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer)

In [ ]:
print(answer.content)

In [ ]:
# repeat the question prompt generation with below question instead of question on allian
# question = "is the topic of nuclear fusion discussed in this video? if yes, then what was discussed?"
# retrieved_docs = retriever.invoke(question)

### Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

In [ ]:
# try the above code with another video and new questions

# Improvements

1. UI based enhancements 
(Final product can be seen as awebsite, for this YTChatbot, we can create chrome plugin use has to install and when he will open YT he can see it and chat using it)

2. Evaluation
   a. Ragas
   b. LangSmith

3. Indexing
   a. Document Ingestion 
   (YT transcript is auto-generated, we can keep a step to fix all bugs in auto-generated transcript; also all videos are not in english or our desired language, we can keep a step to translate transcript from one language to another)

   b. Text Splitting 
   (Instead of character splitter can use semantic chunker)

   c. Vector Store 
   (instead of faiss, can try pinecone)

4. Retrieval
   a. Pre-retrieval
      i. Query rewriting using LLM
      ii. Multi-query generation
      iii. Domain aware routing
   b. Duing Retrieval
      i. MMR
      ii. Hybrid retrieval 
      (Semantic Search + Key-word Search)

      iii. Reranking 
      (creation of ranks of all retrived documents; reranked using LLM)

   c. Post-retrieval
      i. Contextual Compression
      (Can keep only useful part of content, can remove other parts)

5. Augmentation
   a. Prompt Templating 
   b. Answer grounding
   (Don't create answer from yourself, don't halluciante, answer based on fact only)

   c. Context window optimization
   (Trim generated context in such a way that only useful info will stay, otherwise if token limits will cross then LLM will not answer - Is token limit different for input and output?)

6. Generation
   a. Answer with Citation
   b. Guard railing
   (Prevent LLM from generating improper/LLM answers)

7. System Design
   a. Multimodal
   (Can work with text, audio, video etc)

   b. Agentic
   (In process of answering question if needed it can do internet browing apart from understanding context and answer generation)

   c. Memory based
   (Can keep old memory in memory)

These are part of Advanced RAG.